# Microsoft Purview eDiscovery - Case Search & Export Automation

This notebook automates:
1. **App Registration** in Entra AD
2. **App-only Authentication** via Microsoft Graph Python SDK
3. **eDiscovery Case Creation**
4. **Content Search** with KQL queries
5. **Credentials** saved to .env

**Prerequisites:**
- Global Admin access
- .env file with tenant configuration

## 1. Install Dependencies

In [ ]:
import subprocess
import sys

packages = ['azure-identity', 'msgraph-core', 'requests', 'python-dotenv']

for package in packages:
    try:
        __import__(package.replace('-', '_'))
        print(f"✓ {package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package, '-q'])
        print(f"✓ {package} installed")

print("\n✓ All dependencies ready!")

## 2. Load Configuration from .env

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path('.env') if Path('.env').exists() else Path('../.env')

if not env_path.exists():
    raise FileNotFoundError("ERROR: Could not find .env file. Ensure you're running from MW-Toolbox or Compliance directory.")

load_dotenv(env_path)

config = {
    'tenant_id': os.getenv('TENANT_ID'),
    'app_display_name': os.getenv('EDISCOVERY_APP_DISPLAY_NAME'),
    'case_name': os.getenv('EDISCOVERY_CASE_NAME'),
    'search_name': os.getenv('EDISCOVERY_SEARCH_NAME'),
    'search_query': os.getenv('EDISCOVERY_SEARCH_QUERY'),
    'mailbox': os.getenv('EDISCOVERY_MAILBOX_TO_SEARCH'),
    'export_name': os.getenv('EDISCOVERY_EXPORT_NAME'),
}

print("Configuration loaded:")
for key, value in config.items():
    if value is None:
        print(f"  {key}: ⚠️  NOT SET")
    elif 'id' in key.lower():
        print(f"  {key}: {value[:8] if len(str(value)) > 8 else value}...")
    else:
        print(f"  {key}: {value}")

## 3. Authenticate as Admin

In [ ]:
from azure.identity import DeviceCodeCredential
import requests

print("Authenticating as admin for app registration...")
print("(You will be prompted to authenticate in your browser)\n")

credential = DeviceCodeCredential(tenant_id=config['tenant_id'])
token_response = credential.get_token('https://graph.microsoft.com/.default')
headers = {
    'Authorization': f'Bearer {token_response.token}',
    'Content-Type': 'application/json'
}

print("✓ Authenticated successfully")

## 4. Create App Registration

In [ ]:
print(f"Creating app registration: {config['app_display_name']}...\n")

app_body = {
    "displayName": config['app_display_name'],
    "signInAudience": "AzureADMyOrg"
}

response = requests.post(
    'https://graph.microsoft.com/v1.0/applications',
    headers=headers,
    json=app_body
)

if response.status_code != 201:
    print(f"ERROR: {response.status_code} - {response.text}")
    app_id = None
    app_obj_id = None
else:
    app = response.json()
    app_id = app['appId']
    app_obj_id = app['id']
    print(f"✓ App created")
    print(f"  App ID: {app_id[:8]}...")
    print(f"  Object ID: {app_obj_id[:8]}...")

In [ ]:
print("Creating service principal...\n")

sp_body = {"appId": app_id}

response = requests.post(
    'https://graph.microsoft.com/v1.0/servicePrincipals',
    headers=headers,
    json=sp_body
)

if response.status_code != 201:
    print(f"ERROR: {response.status_code} - {response.text}")
else:
    sp = response.json()
    sp_id = sp['id']
    print(f"✓ Service principal created")
    print(f"  SP ID: {sp_id[:8]}...")

In [ ]:
from datetime import datetime, timedelta

print("Generating client secret...\n")

secret_body = {
    "passwordCredential": {
        "displayName": "eDiscovery Demo Secret",
        "endDateTime": (datetime.utcnow() + timedelta(days=365)).isoformat() + "Z"
    }
}

response = requests.post(
    f'https://graph.microsoft.com/v1.0/applications/{app_obj_id}/addPassword',
    headers=headers,
    json=secret_body
)

if response.status_code != 200:
    print(f"ERROR: {response.status_code} - {response.text}")
else:
    secret_response = response.json()
    client_secret = secret_response['secretText']
    print(f"✓ Client secret generated")
    print(f"  Secret: {client_secret[:10]}... ({len(client_secret)} chars)")

## 5. Authenticate as App

In [ ]:
from azure.identity import ClientSecretCredential

print("Authenticating with app credentials...\n")

app_credential = ClientSecretCredential(
    tenant_id=config['tenant_id'],
    client_id=app_id,
    client_secret=client_secret
)

app_token = app_credential.get_token('https://graph.microsoft.com/.default')
app_headers = {
    'Authorization': f'Bearer {app_token.token}',
    'Content-Type': 'application/json'
}

print("✓ App-only authentication successful")

## 6. Create eDiscovery Case

In [ ]:
print(f"Creating eDiscovery case: {config['case_name']}...\n")

case_body = {
    "displayName": config['case_name'],
    "description": "Demo case for automated content search and export",
    "externalId": f"DEMO-CASE-{datetime.now().strftime('%Y%m%d%H%M%S')}"
}

response = requests.post(
    'https://graph.microsoft.com/v1.0/security/cases/ediscoveryCases',
    headers=app_headers,
    json=case_body
)

if response.status_code != 201:
    print(f"ERROR: {response.status_code} - {response.text}")
else:
    case = response.json()
    case_id = case['id']
    print(f"✓ Case created")
    print(f"  Case ID: {case_id[:8]}...")
    print(f"  Status: {case.get('status', 'unknown')}")

## 7. Create Content Search

In [ ]:
print(f"Creating search: {config['search_name']}...\n")

search_body = {
    "displayName": config['search_name'],
    "contentSources": {
        "mailboxes": [config['mailbox']]
    },
    "contentQuery": config['search_query']
}

response = requests.post(
    f'https://graph.microsoft.com/v1.0/security/cases/ediscoveryCases/{case_id}/searches',
    headers=app_headers,
    json=search_body
)

if response.status_code != 201:
    print(f"ERROR: {response.status_code} - {response.text}")
else:
    search = response.json()
    search_id = search['id']
    print(f"✓ Search created")
    print(f"  Search ID: {search_id[:8]}...")
    print(f"  Query: {config['search_query']}")

## 8. Save Credentials to .env

In [ ]:
print("Saving credentials to .env...\n")

env_file = Path(env_path)
env_content = env_file.read_text()

env_content = env_content.replace(
    'EDISCOVERY_APP_CLIENT_ID=""',
    f'EDISCOVERY_APP_CLIENT_ID="{app_id}"'
)
env_content = env_content.replace(
    'EDISCOVERY_APP_CLIENT_SECRET=""',
    f'EDISCOVERY_APP_CLIENT_SECRET="{client_secret}"'
)

env_file.write_text(env_content)

print("✓ Credentials saved to .env")
print(f"  App ID: {app_id[:8]}...")
print(f"  Secret: ({len(client_secret)} characters)")
print("\n⚠️  IMPORTANT: Protect your .env file - it contains sensitive credentials!")

## 9. Summary

In [ ]:
print("\n" + "="*50)
print("eDiscovery Setup Complete!")
print("="*50)
print(f"\nApp Registration:")
print(f"  Name: {config['app_display_name']}")
print(f"  Client ID: {app_id[:8]}...")
print(f"  Secret: (saved to .env)")
print(f"\neDiscovery Case:")
print(f"  Case ID: {case_id[:8]}...")
print(f"  Name: {config['case_name']}")
print(f"\nSearch:")
print(f"  Search ID: {search_id[:8]}...")
print(f"  Name: {config['search_name']}")
print(f"  Query: {config['search_query']}")
print(f"\nNext Steps:")
print(f"  1. Check .env file for saved credentials")
print(f"  2. Complete the search in Compliance Portal")
print(f"  3. Download export when ready")
print("\n" + "="*50)